In [0]:
# Define the storage path
bronze_path = "abfss://bronze@nycyellowtaxistorageacc.dfs.core.windows.net/"
silver_path = "abfss://silver@nycyellowtaxistorageacc.dfs.core.windows.net/"
quanrantine_path = "abfss://quarantine@nycyellowtaxistorageacc.dfs.core.windows.net/"

# Check access to the silver path
try:
    files = dbutils.fs.ls(silver_path)
    display(files)
except Exception as e:
    print(f"Unable to access silver path: {e}")

#Getting data raw data from bronze storage

In [0]:
from pyspark.sql.functions import col, regexp_extract,input_file_name

#extracting all records from bronze layer appending source filename and month
df_raw = spark.read.parquet(bronze_path + "*.parquet").withColumn("source_filename",input_file_name()).withColumn("source_month",regexp_extract("source_filename", r"(\d{4}-\d{2})\.parquet", 1))

print(f"Total number of records: {df_raw.count()}")
display(df_raw)

#Date infiltration filter

In [0]:
from pyspark.sql.functions import year, to_timestamp, lit, when

# Parse timestamps (they come as strings in some years)
df_typed = df_raw.withColumn(
    "pickup_datetime", to_timestamp(col("tpep_pickup_datetime"))
).withColumn(
    "dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime"))
)

# Tag each row with a rejection reason (null = valid)
df_tagged = df_typed.withColumn(
    "reject_reason",
    when(
        year(col("pickup_datetime")) != 2026,
        lit("date_infiltration: pickup year not 2024")
    ).when(
        year(col("dropoff_datetime")) != 2026,
        lit("date_infiltration: dropoff year not 2024")
    ).otherwise(lit(None).cast("string"))
)

df_valid    = df_tagged.filter(col("reject_reason").isNull()).drop("reject_reason")
df_rejected = df_tagged.filter(col("reject_reason").isNotNull())

print(f"Valid rows: {df_valid.count():,}")
print(f"Rejected (date infiltration): {df_rejected.count():,}")
      

#Business rule validation and quarantine write

In [0]:
from pyspark.sql.functions import concat_ws

# Compound validation — every rule gets its own reason string
df_validated = df_valid.withColumn(
    "reject_reason",
    when(col("pickup_datetime") >= col("dropoff_datetime"),
         lit("causality_violation: pickup >= dropoff"))
    .when(col("trip_distance") < 0,
          lit("invalid_distance: negative value"))
    .when(col("fare_amount") < 0,
          lit("invalid_fare: negative amount"))
    .when(col("passenger_count").isNull() | (col("passenger_count") <= 0),
          lit("invalid_passengers: null or zero"))
    .when(col("PULocationID").isNull() | col("DOLocationID").isNull(),
          lit("missing_location_id"))
    .otherwise(lit(None).cast("string"))
)

df_clean    = df_validated.filter(col("reject_reason").isNull()).drop("reject_reason")
df_bad_rows = df_validated.filter(col("reject_reason").isNotNull())

# Combine with previous date-infiltration rejects and write quarantine
df_all_rejects = df_rejected.unionByName(df_bad_rows, allowMissingColumns=True)

(df_all_rejects
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(quanrantine_path)
)

print(f"Clean rows after validation: {df_clean.count():,}")
print(f"Total quarantined rows: {df_all_rejects.count():,}")
display(df_clean)

#Deduplicatoin with window function

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# Deduplicate: one row per unique (pickup_time, dropoff_time, PULocationID, DOLocationID, fare_amount)
dedup_window = Window.partitionBy(
    "pickup_datetime", "dropoff_datetime",
    "PULocationID", "DOLocationID", "fare_amount"
).orderBy("pickup_datetime")  # keep earliest source if duplicated across months

df_deduped = (
    df_clean
    .withColumn("row_num", row_number().over(dedup_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

dupes_removed = df_clean.count() - df_deduped.count()
print(f"Duplicate rows removed: {dupes_removed:,}")

#Derived coolumns (feature engineering)

In [0]:
from pyspark.sql.functions import (
    round as spark_round, unix_timestamp, dayofweek,
    hour, month, date_format, expr
)

df_enriched = (
    df_deduped

    # Trip duration in minutes
    .withColumn(
        "trip_duration_min",
        spark_round(
            (unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60,
            2
        )
    )

    # Average speed in mph (guard against zero duration)
    .withColumn(
        "avg_speed_mph",
        when(
            col("trip_duration_min") > 0,
            spark_round(col("trip_distance") / (col("trip_duration_min") / 60), 2)
        ).otherwise(lit(None).cast("double"))
    )

    # Time dimension extracts
    .withColumn("pickup_hour",        hour("pickup_datetime"))
    .withColumn("pickup_day_of_week", dayofweek("pickup_datetime"))  # 1=Sun, 7=Sat
    .withColumn("pickup_month",       month("pickup_datetime"))
    .withColumn("pickup_date",        col("pickup_datetime").cast("date"))

    # Is it a weekend?
    .withColumn(
        "is_weekend",
        col("pickup_day_of_week").isin([1, 7]).cast("boolean")
    )

    # Total trip cost (fare + all extras)
    .withColumn(
        "total_cost",
        spark_round(
            col("fare_amount") + col("extra") + col("mta_tax") +
            col("tip_amount") + col("tolls_amount") + col("improvement_surcharge"),
            2
        )
    )

    # Tip percentage
    .withColumn(
        "tip_pct",
        when(
            col("fare_amount") > 0,
            spark_round(col("tip_amount") / col("fare_amount") * 100, 1)
        ).otherwise(lit(None).cast("double"))
    )

    # Payment type as readable label
    .withColumn(
        "payment_type_label",
        when(col("payment_type") == 1, "credit_card")
        .when(col("payment_type") == 2, "cash")
        .when(col("payment_type") == 3, "no_charge")
        .when(col("payment_type") == 4, "dispute")
        .otherwise("unknown")
    )

    # Speed outlier flag (>100 mph is physically implausible in NYC)
    .withColumn(
        "is_speed_outlier",
        (col("avg_speed_mph") > 100).cast("boolean")
    )
)

#Zone lookup join (enrichment)


In [0]:
ZONE_LOOKUP_PATH = "abfss://bronze@nycyellowtaxistorageacc.dfs.core.windows.net/lookup/taxi_zone_lookup.csv"

# Load and prep the lookup table
df_zones = (
    spark.read.csv(ZONE_LOOKUP_PATH, header=True, inferSchema=True)
    .select(
        col("LocationID").cast("integer"),
        col("Borough").alias("borough"),
        col("Zone").alias("zone_name"),
        col("service_zone")
    )
)

# Join twice — once for pickup, once for dropoff
df_silver = (
    df_enriched
    .join(
        df_zones.alias("pu_zone"),
        col("PULocationID") == col("pu_zone.LocationID"),
        how="left"
    )
    .withColumnRenamed("borough",      "pickup_borough")
    .withColumnRenamed("zone_name",    "pickup_zone")
    .withColumnRenamed("service_zone", "pickup_service_zone")
    .drop("LocationID")

    .join(
        df_zones.alias("do_zone"),
        col("DOLocationID") == col("do_zone.LocationID"),
        how="left"
    )
    .withColumnRenamed("borough",      "dropoff_borough")
    .withColumnRenamed("zone_name",    "dropoff_zone")
    .withColumnRenamed("service_zone", "dropoff_service_zone")
    .drop("LocationID")
)

# Write Silver as Delta, partitioned by month for query performance
(df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_month")
    .save(silver_path)
)

print(f"Silver layer written. Final row count: {df_silver.count():,}")
print(f"Partitions: pickup_month (Jan–Dec 2024)")

In [0]:
%sh
find / -name "stg_yellow_trips.sql" 2>/dev/null